# Spatial Autocorrelation and Spatial Regression

Data in social science are almost never "independently and identically distributed": unemployment rates, house prices, voting preferences, pandemic spread——**nearby places resemble each other**. When one county has high unemployment, its neighboring counties typically do too. Ignoring this spatial dependence, ordinary OLS underestimates standard errors, misattributing phenomena that belong to "diffusion/spillover" as "individual effects", rendering inference unreliable. Spatial analysis is a set of methods that formally incorporates "geographic proximity relationships" into models.

Complete spatial analysis typically proceeds in two steps, which this tutorial follows. **The first step is diagnosis**: does the variable cluster in space? We first use **global Moran's I**——a scalar plus a permutation test yielding a pseudo p-value——to answer "yes or no"; then use **local LISA** to decompose this global statistic to each location, answering "where is it clustered, hotspot or cold spot". **The second step is modeling**: if clustering is real, we use a **spatial lag model (SAR)** `y = ρ·Wy + Xβ + ε` to incorporate neighbors' effects into the equation, estimate the spatial autoregressive coefficient `ρ`, and decompose the predictor's effect into direct, indirect (spatial spillover), and total effects——something ordinary regression cannot provide.

The key object throughout is the **spatial weights matrix `W`**: it encodes "who is whose neighbor" and is the only new thing distinguishing spatial analysis from ordinary tabular analysis. This tutorial uses `socialverse` to walk through the full chain; it is an analysis library designed for social science that registers each method in a function registry, validates readiness at runtime, and automatically builds a reproducible evidence chain. Methodologically and implementationally, global and local autocorrelation here parallel Python's **PySAL `esda`** and R's **`spdep`**; spatial lag regression parallels PySAL's **`spreg.ML_Lag`** and R's **`spatialreg::lagsarlm`**.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import numpy as np
import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

## Loading Spatial Data

We use a built-in synthetic spatial dataset: an 8×8 **rook adjacency grid** with 64 points in total (imagine 64 counties arranged in a square matrix, where up-down and left-right neighbors are considered neighbors). The data are generated by a true SAR process with **true spatial autoregressive coefficient ρ = 0.5 and predictor coefficient β_x = 1.0**. With known true values, this tutorial has a falsifiable target: our estimated `ρ` should fall near 0.5, and whether the method can recover it is a hard test.

`ds.load_spatial` returns a `(df, W)` **tuple**: each row in `df` represents one grid point, with columns `id, row, col` (location), `y` (outcome), and `x` (predictor); `W` is the 64×64 adjacency matrix **row-standardized**——each row sums to 1, so `Wy` is the "average y of neighbors". Corner points have 2 neighbors, edge points 3, and interior points 4.

In [2]:
df, W = ds.load_spatial(rho=0.5)   # SAR grid,真实 rho = 0.5

print(f"格点数 n = {len(df)}   ·   权重矩阵 W 形状 = {W.shape}")
df.head(6)

格点数 n = 64   ·   权重矩阵 W 形状 = (64, 64)


,id,row,col,y,x
0,0,0,0,0.0732,0.1257
1,1,0,1,-0.2868,-0.1321
2,2,0,2,1.5382,0.6404
3,3,0,3,0.7627,0.1049
4,4,0,4,-0.7016,-0.5357
5,5,0,5,-0.6585,0.3616


Check two basic properties of `W`: each row sums to 1 (by definition of row standardization), and the number of neighbors varies by location——fewer at corners, more at interior points. These confirm that `W` is a properly row-standardized rook adjacency matrix.

In [3]:
row_sums = W.sum(axis=1)                    # 每行和
neighbours = (W > 0).sum(axis=1)            # 每个格点的邻居数
print(f"W 每行和(应恒为 1):min={row_sums.min():.3f}, max={row_sums.max():.3f}")
print("邻居数分布(角=2, 边=3, 内=4):",
      dict(zip(*np.unique(neighbours, return_counts=True))))

W 每行和(应恒为 1):min=1.000, max=1.000
邻居数分布(角=2, 边=3, 内=4): {np.int64(2): np.int64(4), np.int64(3): np.int64(24), np.int64(4): np.int64(36)}


## Establishing a Study State

The starting point of analysis is to register the data into a `StudyState`. Spatial functions recognize the `(df, W)` packaging: we write the entire tuple into `sources.datasets` once, and every subsequent step——diagnosis, modeling, plotting——reads data and the weights matrix from here, **without needing to pass `W` repeatedly**.

In [4]:
st = sv.StudyState()
st.write("sources", "datasets", (df, W))   # (df, W) 元组即整条链的单一数据源
print(repr(st))

StudyState(sources[1]; 0 steps)


## Global Moran's I: Does the Variable Cluster in Space?

Diagnose before modeling. **Global Moran's I** is a scalar measuring whether adjacent points are correlated: positive values indicate high near high and low near low (positive spatial autocorrelation), near zero suggests spatial randomness, negative values indicate checkerboard alternation. Its significance is given by a **permutation test**——randomly shuffle the labels of `y` across grid points 999 times, obtain a reference distribution of "what we would see without spatial structure", then see how extreme the true I is. This is exactly what `esda.Moran(permutations=999)` does.

`spatial_autocorr` in a single call computes both global Moran and the local LISA needed later. `value="y"` specifies which variable to diagnose, and `seed=0` fixes the random seed of the permutation to ensure reproducibility.

In [5]:
sv.tl.spatial_autocorr(st, value="y", permutations=999, seed=0)

moran = st.diagnostics["moran"]
print(f"Moran's I   : {moran['I']:.4f}   (n = {moran['n']})")
print(f"E[I]        : {moran['expected_I']:.4f}   (空间随机时的期望)")
print(f"z 分数      : {moran['z_score']:.3f}")
print(f"置换 p 值   : {moran['p_perm']:.4f}   ({moran['permutations']} 次置换,双侧)")
print(f"显著象限计数: {moran['cluster_counts']}   (p<0.05)")

Moran's I   : 0.3894   (n = 64)
E[I]        : -0.0159   (空间随机时的期望)
z 分数      : 4.345
置换 p 值   : 0.0010   (999 次置换,双侧)
显著象限计数: {'HH': 4, 'LH': 0, 'LL': 1, 'HL': 0}   (p<0.05)


`I ≈ 0.39`, far greater than the expectation under spatial randomness `E[I] ≈ −0.016`, and permutation p-value = 0.001——**strong and significant positive spatial autocorrelation**: high-value points cluster together, low-value points cluster together. This is perfectly consistent with the ρ = 0.5 positive spatial dependence built into the data-generating process. The significant quadrants are dominated by HH (hotspots) and LL (cold spots), indicating "clustering" rather than "isolation".

## Local LISA: Where Does Clustering Occur?

Global I only answers "yes or no"; **LISA** (Local Indicators of Spatial Association) further answers "where". It decomposes global Moran to each grid point: each point gets a local `I_i`, a Moran scatter quadrant (HH=hotspot, LL=cold spot, HL/LH=spatial outlier), and a conditional permutation p-value. These results were already computed in the previous step and are stored in `st.models["lisa"]`, ready to use directly.

Below we organize each point's local results into a table and pick out the 8 points with the strongest `|I_i|`——they are where spatial structure is most prominent.

In [6]:
lisa = st.models["lisa"]
print(f"显著局部簇的格点数(p_sim<0.05):{lisa['n_significant']} / {len(df)}")
print(f"各象限显著计数:{lisa['cluster_counts']}   (HH=热点, LL=冷点, HL/LH=离群)")

lisa_df = pd.DataFrame({
    "id": df["id"], "row": df["row"], "col": df["col"], "y": df["y"],
    "Ii": np.round(lisa["Ii"], 3),
    "quadrant": lisa["quadrant"],
    "p_sim": np.round(lisa["p_sim"], 3),
})
# 按局部 I_i 的绝对值排序,取最突出的 8 个格点
lisa_df.reindex(lisa_df["Ii"].abs().sort_values(ascending=False).index).head(8)

显著局部簇的格点数(p_sim<0.05):5 / 64
各象限显著计数:{'HH': 4, 'LH': 0, 'LL': 1, 'HL': 0}   (HH=热点, LL=冷点, HL/LH=离群)


,id,row,col,y,Ii,quadrant,p_sim
47,47,5,7,2.8127,2.822,HH,0.031
56,56,7,0,1.8262,2.275,HH,0.018
59,59,7,3,-1.9838,2.271,LL,0.024
49,49,6,1,2.1879,2.074,HH,0.015
12,12,1,4,-2.9998,2.071,LL,0.110
48,48,6,0,2.4018,1.846,HH,0.082
39,39,4,7,1.7174,1.564,HH,0.037
60,60,7,4,-1.0097,1.063,LL,0.050


Among the 64 grid points, 5 reach local significance at `p_sim < 0.05`, concentrated in the HH and LL quadrants——hotspots cluster together, cold spots cluster together. The strongest points (such as `id=47` with `I_i ≈ 2.82`) all fall in HH/LL, consistent with the global diagnosis direction: this is a spatial field dominated by positive association.

## Moran Scatterplot: Visualizing the Diagnosis

The Moran scatterplot visualizes global I: the horizontal axis is the standardized value `z`, the vertical axis is its spatial lag `Wz` (average of neighbors), **the slope of the fitted line exactly equals Moran's I**. The four quadrants are clear at a glance——upper right HH (hotspots), lower left LL (cold spots), lower right HL and upper left LH (spatial outliers). It and the scalar I above are two presentations of the same thing: one gives significance numbers, the other gives spatial intuition.

`sv.pl.moran_scatter` reads the diagnosis results and `(df, W)` back from the study state, reconstructs `z` and `Wz`, and plots. The figure is saved as a PNG in the same directory, then referenced in markdown.

In [7]:
sv.pl.moran_scatter(st, variable="y", out="fig_moran.png")
print("图已保存:fig_moran.png")

图已保存:fig_moran.png


![Moran Scatterplot · Fitted slope = Moran's I](fig_moran.png)

The point cloud tilts from lower left to upper right, with fitted slope ≈ reported `I ≈ 0.39`——**positive slope means positive spatial autocorrelation**. The vast majority of points fall in HH/LL (positive association) quadrants, with only scattered points in HL/LH (spatial outliers), perfectly matching the quadrant counts above.

## Declaring the Outcome Variable

Diagnosis confirms strong spatial dependence, and now we model. The outcome in spatial regression is part of the research design and should not be guessed by the function——so before running SAR, explicitly declare which variable is the outcome. This step writes `y` into the study state's `variables.outcome` slot, which `spatial_regression` will read from.

In [8]:
st.write("variables", "outcome", "y")   # 显式声明因变量
print(repr(st))

StudyState(sources[1], variables[1], models[1], diagnostics[1], artifacts[1]; 2 steps)


## Spatial Lag Regression (SAR): Estimating ρ

Now we incorporate spatial dependence into the model: `y = ρ·Wy + Xβ + ε`. Here `Wy` is the "average outcome of neighbors", and the coefficient `ρ` measures **how much a point's outcome is pulled by its neighbors' outcomes**. Estimation uses **concentrated likelihood ML**: first use OLS to concentrate out `β` and `σ²`, leaving a one-dimensional objective in `ρ`, then maximize using SciPy's bounded search——the objective includes the `log|I − ρW|` Jacobian term (computed via `W`'s eigenvalues), exactly the algorithm of `spreg.ML_Lag`.

We still need not pass `W`——it is automatically extracted from the `(df, W)` tuple in `sources.datasets`. `outcome="y"` matches the outcome just declared, and `predictors=["x"]` specifies the predictor.

In [9]:
sv.tl.spatial_regression(st, outcome="y", predictors=["x"])

sar = st.models["sar"]
print("=== 空间滞后模型 (SAR):y = rho * Wy + X beta + eps ===")
print(f"  样本量 n   : {sar['n']}")
print(f"  rho        : {sar['rho']:.4f}     ← 真实 rho = 0.5")
print(f"  beta       : const={sar['beta']['const']:.4f}, "
      f"x={sar['beta']['x']:.4f}   ← 真实 beta_x = 1.0")
print(f"  sigma^2    : {sar['sigma2']:.4f}")
print(f"  对数似然   : {sar['loglik']:.3f}")

=== 空间滞后模型 (SAR):y = rho * Wy + X beta + eps ===
  样本量 n   : 64
  rho        : 0.5299     ← 真实 rho = 0.5
  beta       : const=0.0307, x=0.8985   ← 真实 beta_x = 1.0
  sigma^2    : 0.2341
  对数似然   : 43.581


**The method recovers known parameters.** Estimated `ρ ≈ 0.53` (true 0.5) and `β_x ≈ 0.90` (true 1.0)——on a small sample of just 64 grid points, concentrated likelihood ML stably recovers both data-generating parameters. This is the value of synthetic data with known parameters: method correctness requires no faith, only observation.

## Direct / Indirect / Total Effects

In SAR, `β_x` is **not** the complete marginal effect of `x` on `y`. The reason is that a change in `x` at one point changes its own `y`, which then spills over to neighbors via `Wy`, and feeds back. The truly interpretable quantities come from the reduced form `(I − ρW)⁻¹` (LeSage & Pace decomposition): **direct effect** is the impact of a point's own `x` change on its own `y` (including spatial feedback), **indirect effect** is the sum of spillovers to all neighbors, **total effect** is the sum of the two.

In [10]:
impacts = sar["impacts"]["x"]
print("=== x 的效应分解 (LeSage-Pace, 基于 (I - rho W)^-1) ===")
print(f"  直接效应 (direct)   : {impacts['direct']:.4f}   (含自我空间反馈)")
print(f"  间接效应 (indirect) : {impacts['indirect']:.4f}   (溢出到邻居)")
print(f"  总效应   (total)    : {impacts['total']:.4f}   (≈ beta_x / (1 - rho))")

=== x 的效应分解 (LeSage-Pace, 基于 (I - rho W)^-1) ===
  直接效应 (direct)   : 0.9882   (含自我空间反馈)
  间接效应 (indirect) : 0.9231   (溢出到邻居)
  总效应   (total)    : 1.9113   (≈ beta_x / (1 - rho))


The indirect effect (spatial spillover) is clearly nonzero and positive: `ρ>0` means that an increase in `x` at one place spills over to neighbors' `y`. Total effect ≈ `β_x/(1−ρ) ≈ 0.9/0.47 ≈ 1.9`, nearly double the direct effect. **Using ordinary OLS alone, you only see β and completely miss this spillover**——this is why spatial econometrics exists.

## Reproducible Evidence Chain

Finally, note the key difference between `socialverse` and ordinary spatial analysis scripts. The entire chain starts from the `(df, W)` tuple, passes through Moran diagnosis, LISA, Moran scatterplot, and SAR estimation, with all results landing in **named slots of a single `StudyState`**; every successful function call automatically welds a record——"which function was used, what was consumed, what was produced"——into the evidence chain. Compared to PySAL/spdep, you need to manually glue together `esda` + `spreg` + `libpysal` + `matplotlib` and keep your own accounting; here, accounting is a byproduct of analysis.

In [11]:
print(st.summary())

StudyState
  sources: datasets
  variables: outcome
  models: lisa, sar
  diagnostics: moran, spatial
  artifacts: figures
  provenance: 3 step(s)


## Summary

We have completed a standard spatial analysis chain: load spatial data with `W` → global Moran diagnosis → local LISA localization → Moran scatterplot → declare outcome variable → SAR estimate ρ → effect decomposition. It parallels Python's **PySAL** (`esda` for autocorrelation, `spreg.ML_Lag` for spatial lag regression) and R's **`spdep` / `spatialreg`**.

Compared to pure estimation tools, here we have two additional things: all steps share **a single study state** (slots produced by diagnosis are read directly by plotting and modeling, no need to manually move data), and an **evidence chain** woven throughout by the registry. The next tutorial [15_qca_demography](15_qca_demography.ipynb) turns to fsQCA configurational analysis and demographic life tables/decomposition methods.